# Множества и словари — 12.11.2026

Материал на одно занятие. Цель — научиться хранить уникальные значения, находить общие элементы, получать данные по ключу и вести счётчики.

1. [Множества: создание и основные операции](#sets).
2. [Словари: ключи, значения и перебор](#dicts).
3. [Хешируемость: что можно хранить в множестве и использовать как ключ](#hashability).
4. [Скорость основных операций](#complexity).
5. [Шпаргалка и вопросы для самопроверки](#summary).

Запускай примеры по порядку. После каждого примера попробуй объяснить результат своими словами.

<a id="sets"></a>
## 1. Множество — `set`

### Зачем нужно множество

Множество хранит **уникальные значения**. Например, один посетитель мог зайти на сайт несколько раз, но множество его имён позволит узнать число разных посетителей.

Главные свойства:

- Равные элементы не дублируются.
- Порядок не гарантируется: нельзя рассчитывать ни на сортировку, ни на порядок добавления.
- Индексов нет. Запись `visitors[0]` вызовет `TypeError`.
- Содержимое можно менять: добавлять и удалять элементы.
- Элементы должны быть хешируемыми. Пока работаем со строками и числами; подробнее об ограничениях — в разделе 3.

### Создание и проверка наличия

Непустое множество записывают в фигурных скобках без двоеточий. `set(данные)` создаёт множество из перебираемых данных, убирая повторы. Пустое множество — **`set()`**, а `{}` — пустой словарь.

`len(s)` — число уникальных элементов. `x in s` проверяет наличие, `x not in s` — отсутствие.

In [ ]:
visitors = {"Лена", "Илья", "Лена"}
print(visitors == {"Илья", "Лена"})  # True — порядок не влияет на равенство
print(len(visitors))                  # 2
print("Лена" in visitors)             # True
print("Олег" not in visitors)         # True

visits = ["Лена", "Илья", "Лена", "Олег"]
unique_visitors = set(visits)
print(len(unique_visitors))  # 3
print(visits)  # исходные данные не изменились

print(set("мама") == {"м", "а"})  # True — строка перебирается по символам
print(type(set()))  # <class 'set'>
print(type({}))     # <class 'dict'>

### Добавление и удаление

| Действие | Запись | Особенность |
|---|---|---|
| Добавить один элемент | `s.add(x)` | Если он уже есть, ничего не изменится |
| Добавить элементы из другой коллекции | `s.update(values)` | Добавляется каждый элемент из `values` |
| Удалить элемент без ошибки при отсутствии | `s.discard(x)` | Отсутствующий элемент игнорируется |
| Удалить обязательный элемент | `s.remove(x)` | При отсутствии будет `KeyError` |
| Удалить всё | `s.clear()` | Само множество остаётся, но становится пустым |

Эти методы меняют само множество и возвращают `None`. Поэтому `s = s.add(x)` — ошибка: переменная `s` станет равна `None`.

Отличай `add()` от `update()`: `s.add("кот")` добавит одну строку, а `s.update("кот")` добавит отдельные символы строки.

In [ ]:
members = {"Лена", "Илья"}
members.add("Олег")
members.add("Лена")  # повтор не добавится
members.update(["Нина", "Олег"])
members.discard("Илья")
members.discard("Незнакомец")  # ошибки нет
print(sorted(members))  # ['Лена', 'Нина', 'Олег']

# Ошибку ловим специально, чтобы пример не останавливал всё занятие.
try:
    members.remove("Незнакомец")
except KeyError:
    print("remove: такого элемента нет")

whole_word = set()
whole_word.add("кот")
letters = set()
letters.update("кот")
print(whole_word == {"кот"})  # True
print(letters == {"к", "о", "т"})  # True

### Операции над двумя множествами

Представим два кружка: рисование и музыка. По спискам участников можно узнать, кто ходит в оба кружка, хотя бы в один или только в один из них.

| Операция | Запись | Что попадёт в ответ |
|---|---|---|
| Пересечение | `a & b` | Есть и в `a`, и в `b` |
| Объединение | `a | b` | Есть хотя бы в одном |
| Разность | `a - b` | Есть в `a`, но нет в `b` |
| Симметрическая разность | `a ^ b` | Есть ровно в одном из двух |

Все четыре операции создают **новое множество**, исходные не меняют. Разность зависит от порядка операндов.

Для двух входных списков сначала создай множества: `set(first) & set(second)`. Между двумя списками оператор `&` не работает.

In [ ]:
drawing = {"Лена", "Илья", "Олег"}
music = {"Илья", "Нина"}

print(sorted(drawing & music))  # ['Илья']
print(sorted(drawing | music))  # ['Илья', 'Лена', 'Нина', 'Олег']
print(sorted(drawing - music))  # ['Лена', 'Олег']
print(sorted(music - drawing))  # ['Нина']
print(sorted(drawing ^ music))  # ['Лена', 'Нина', 'Олег']
assert drawing == {"Лена", "Илья", "Олег"}  # исходное множество сохранено

### Подмножество, перебор и вывод

`a <= b` проверяет, входят ли **все элементы** `a` в `b`. Это проверка подмножества. `a < b` дополнительно требует, чтобы множества не были равны.

Не путай «элемент входит в множество» (`x in s`) с «все элементы одного множества входят в другое» (`a <= b`).

Цикл `for` перебирает элементы множества, но порядок не гарантируется. Для вывода по порядку используй `sorted(s)`: она возвращает новый **список**, не меняя множество. В примерах ниже сортируем только строки или только числа, чтобы элементы были сравнимы.

In [ ]:
required = {"Python", "Git"}
known = {"Python", "Git", "SQL"}
print("Python" in known)  # True — проверка одного элемента
print(required <= known)  # True — проверка всех требований
print(required < known)   # True — в known есть и другие элементы
print(known < known)      # False

for skill in sorted(known):
    print(skill)  # Git, Python, SQL — по одному на строку

<a id="dicts"></a>
## 2. Словарь — `dict`

### Ключ → значение

Множество отвечает на вопрос «есть ли элемент?». Словарь позволяет по одному значению получить другое: по названию товара — цену, по имени — баллы, по слову — число его появлений.

Словарь записывают в фигурных скобках, разделяя **ключ и значение двоеточием**: `{"тетрадь": 50}`. Пустой словарь — `{}` или `dict()`.

- Ключи уникальны и должны быть хешируемыми.
- Значения могут повторяться и могут быть любого типа, включая списки и другие словари.
- `len(d)` возвращает количество пар, то есть число ключей.
- Словарь сохраняет порядок добавления ключей. Это не автоматическая сортировка.

Одинаковый ключ не создаёт вторую запись: более позднее значение заменяет старое.

In [ ]:
prices = {"тетрадь": 50, "ручка": 30, "карандаш": 30}
print(len(prices))         # 3 — одинаковые цены допустимы
print(prices["тетрадь"])  # 50 — обращаемся по ключу

prices["ластик"] = 20  # новый ключ — добавление
prices["ручка"] = 35   # существующий ключ — замена
print(prices)
# {'тетрадь': 50, 'ручка': 35, 'карандаш': 30, 'ластик': 20}

repeated_key = {"баллы": 10, "баллы": 25}
print(repeated_key)  # {'баллы': 25}

### Получение значения и отсутствие ключа

`d[key]` подходит, когда ключ обязан быть в словаре. Если его нет, Python выдаст `KeyError`.

Если отсутствие ключа — обычная ситуация, используй `d.get(key, default)`. Метод вернёт найденное значение или `default`, если ключа нет. Если второй аргумент опущен, запасной ответ — `None`.

Важно: `.get()` **не добавляет** ключ и не заменяет значения `0`, `False`, `""` или `None` запасным ответом. Он проверяет именно наличие ключа.

In [ ]:
stock = {"тетрадь": 4, "ручка": 0, "ластик": None}
print(stock["тетрадь"])          # 4
print(stock.get("линейка", 0))   # 0 — ключа нет
print(stock.get("линейка"))      # None
print(stock.get("ручка", 100))   # 0 — ключ есть
print(stock.get("ластик", 100))  # None — это записанное значение
print("линейка" in stock)        # False — get ничего не добавил

try:
    print(stock["линейка"])
except KeyError:
    print("Такого ключа нет")

### Что проверяет `in` и как перебирать словарь

`key in d` проверяет **ключи**, а не значения. Чтобы искать среди значений, пиши `value in d.values()`.

| Запись | Что перебирает |
|---|---|
| `for key in d` | Ключи |
| `for key in d.keys()` | Тоже ключи |
| `for value in d.values()` | Значения |
| `for key, value in d.items()` | Пары «ключ, значение» |

В последнем варианте Python разбирает каждую пару в две переменные. Названия переменных выбирай по смыслу: `product, price`, `name, score`.

Не добавляй и не удаляй ключи того же словаря во время перебора: это может вызвать `RuntimeError`. Для создания результата удобно завести отдельный словарь.

In [ ]:
prices = {"тетрадь": 50, "ручка": 30}
print("ручка" in prices)       # True
print(30 in prices)            # False — такого ключа нет
print(30 in prices.values())   # True

for product in prices:
    print(product)  # тетрадь, затем ручка

for product, price in prices.items():
    print(f"{product}: {price} руб.")

# Новый словарь: стоимость трёх штук каждого товара.
three_prices = {}
for product, price in prices.items():
    three_prices[product] = price * 3
print(three_prices)  # {'тетрадь': 150, 'ручка': 90}
print(prices)        # {'тетрадь': 50, 'ручка': 30}

### Удаление и обновление

- `del d[key]` удаляет пару. Если ключа нет, будет `KeyError`.
- `d.pop(key, default)` удаляет пару и возвращает прежнее значение. Если ключа нет, возвращает `default`. Без запасного ответа отсутствие ключа вызовет `KeyError`.
- `d.update(other)` добавляет пары из другого словаря. Совпадающие ключи получают новые значения.
- `d.clear()` удаляет все пары.

`update()` меняет словарь и возвращает `None`. Повторная запись значения существующего ключа не меняет его место в порядке перебора.

In [ ]:
prices = {"тетрадь": 50, "ручка": 30}
prices.update({"ручка": 35, "ластик": 20})
print(prices)  # {'тетрадь': 50, 'ручка': 35, 'ластик': 20}
removed_price = prices.pop("ластик", 0)
print(removed_price)             # 20
print(prices.pop("линейка", 0))  # 0 — ключа не было

del prices["ручка"]
print(prices)  # {'тетрадь': 50}
prices.clear()
print(prices)  # {}

### Как работает счётчик

Словарь позволяет хранить отдельный счётчик для каждого ключа. Разберём увеличение одного счётчика:

1. Получить старое значение: `counts.get(key, 0)`. Для нового ключа берём ноль.
2. Прибавить единицу.
3. Записать результат обратно: `counts[key] = ...`.

Например, при ручной регистрации посещений ключ — название дня. Первое посещение создаёт значение `1`, второе увеличивает его до `2`.

Если нужно обработать много событий, этот же шаг повторяют в цикле для каждого события. В ДЗ ты самостоятельно соберёшь такой цикл.

In [ ]:
visits = {}

day = "понедельник"
old_count = visits.get(day, 0)
visits[day] = old_count + 1
print(visits)  # {'понедельник': 1}

# То же увеличение короче:
visits[day] = visits.get(day, 0) + 1
print(visits)  # {'понедельник': 2}

visits["вторник"] = visits.get("вторник", 0) + 1
print(visits)  # {'понедельник': 2, 'вторник': 1}

### Как сохранить исходную коллекцию

И множества, и словари изменяемы. Присваивание `other = original` создаёт ещё одно имя **того же объекта**. Изменение через одно имя будет видно через другое.

Для отдельной копии есть `.copy()`. Если нужно преобразовать все значения словаря, можно создать `{}` и заполнить его циклом, как в примере со стоимостью трёх товаров.

Копия через `.copy()` поверхностная: вложенные изменяемые объекты остаются общими. Для наших примеров со строками и числами достаточно обычной копии.

In [ ]:
original = {"жетоны": 5}
alias = original
alias["жетоны"] = 8
print(original)  # {'жетоны': 8} — объект один

separate = original.copy()
separate["жетоны"] = 100
print(original)  # {'жетоны': 8}
print(separate)  # {'жетоны': 100}

first = {1, 2}
second = first.copy()
second.add(3)
print(first == {1, 2})      # True
print(second == {1, 2, 3})  # True

<a id="hashability"></a>
## 3. Хешируемость и ограничения элементов и ключей

### Что означает «хешируемый»

Для быстрого поиска множество и словарь используют **хеш** — целое число, вычисляемое для объекта. Получить его можно функцией `hash(value)`.

Объект хешируем, если у него есть хеш, который не меняется в течение жизни объекта, и соблюдается правило: **равные объекты имеют одинаковые хеши**. Обратный вывод делать нельзя: одинаковые хеши сами по себе не доказывают равенство объектов.

Для работы со встроенными типами достаточно практических правил ниже. Устройство хеш-таблиц здесь не разбираем.

| Тип значения | Элемент `set` | Ключ `dict` | Значение `dict` |
|---|---|---|---|
| `int`, `str`, `bool`, `None` | Да | Да | Да |
| Кортеж из хешируемых элементов, например `(2, 5)` | Да | Да | Да |
| Кортеж со списком внутри, например `(2, [5])` | Нет | Нет | Да |
| `list`, `dict`, `set` | Нет | Нет | Да |

Обычные изменяемые контейнеры `list`, `dict` и `set` не хешируемы. Не запоминай правило «любой неизменяемый объект хешируем»: например, кортеж сам неизменяем, но со списком внутри хешируемым не будет.

Ограничение относится к **элементам множества и ключам словаря**. Для значений словаря такого требования нет.

In [ ]:
print(hash(42))  # 42
print(hash("Python") == hash("Python"))  # True в одном процессе

# Конкретное число хеша строки запоминать не нужно:
# при другом запуске Python оно может отличаться.

locations = {(1, 2), (3, 4)}  # кортежи из чисел допустимы
labels = {(1, 2): "старт", (3, 4): "финиш"}
print((1, 2) in locations)  # True
print(labels[(1, 2)])       # старт

# Список разрешён как ЗНАЧЕНИЕ словаря.
student = {"имя": "Лена", "оценки": [5, 4, 5]}
student["оценки"].append(5)
print(student["оценки"])  # [5, 4, 5, 5]

### Как выглядит ошибка хешируемости

Попытка вычислить хеш списка или использовать его как ключ приводит к `TypeError: unhashable type: 'list'`. Аналогичное ограничение действует для словарей и обычных множеств.

В примере ошибки намеренно перехвачены через `try / except`, чтобы остальные ячейки можно было выполнить. Для решения ДЗ перехватывать такие ошибки не требуется — нужно выбирать допустимые типы.

In [ ]:
for value in ([1, 2], {"x": 1}, {1, 2}, (1, [2])):
    try:
        hash(value)
    except TypeError as error:
        print(type(value).__name__, "→", error)

try:
    invalid_set = {[1, 2]}  # список как элемент множества
except TypeError:
    print("Список нельзя добавить как элемент множества")

try:
    invalid_dict = {[1, 2]: "точка"}  # список как ключ
except TypeError:
    print("Список нельзя использовать как ключ словаря")

### Уникальность определяется равенством

Для множества равные значения считаются одним элементом, для словаря — одним ключом. Типы при этом не обязательно одинаковы: `1`, `1.0` и `True` равны друг другу в Python и имеют одинаковые хеши.

Поэтому не стоит смешивать логические значения и числа в роли разных идентификаторов. Строка `"1"` при этом не равна числу `1`.

In [ ]:
print(1 == 1.0 == True)  # True
print(len({1, 1.0, True}))  # 1

same_key = {1: "число", True: "логическое значение"}
print(len(same_key))    # 1
print(same_key[1])      # логическое значение — прежнее значение заменилось
print(len({1, "1"}))    # 2

<a id="complexity"></a>
## 4. Скорость операций

### Что означают `O(1)` и `O(n)`

Здесь `n` — число элементов коллекции. Оценка описывает, как растёт объём работы при увеличении данных, а не точное число секунд.

- **`O(1)`**: объём работы в среднем почти не зависит от количества элементов.
- **`O(n)`**: в общем случае нужно просмотреть количество элементов, пропорциональное размеру коллекции.

При поиске `x in items` в списке Python сравнивает элементы по очереди, пока не найдёт нужный или не дойдёт до конца. Если значения нет, просматривается весь список: `O(n)`.

Множество и словарь используют хеш для поиска, поэтому проверка элемента множества или ключа словаря обычно обходится без полного перебора: **в среднем `O(1)`**.

### Основные оценки для CPython

| Операция | Оценка |
|---|---|
| `x in s`, где `s` — множество | В среднем `O(1)` |
| `s.add(x)`, `s.discard(x)`, `s.remove(x)` | В среднем `O(1)` |
| `key in d`, `d[key]`, `d.get(key)` | В среднем `O(1)` |
| `d[key] = value`, удаление одного ключа | В среднем `O(1)` |
| `value in d.values()` | `O(n)` — ищем среди значений |
| Полный перебор множества или словаря | `O(n)` |
| Создание множества из `n` входных элементов | В среднем `O(n)` |
| Копирование множества или словаря | `O(n)` |
| `len(s)` или `len(d)` | `O(1)` |

Оценки поиска, добавления и удаления предполагают, что хеширование и сравнение отдельного элемента не требуют растущего времени. Для длинных строк вычисление хеша тоже имеет свою стоимость. «В среднем» не означает гарантии для каждого вызова: в неблагоприятном случае поиск в множестве или словаре может занять `O(n)`. Для наших задач достаточно средних оценок.

Операция над **всем множеством** не становится `O(1)` только потому, что поиск одного элемента быстрый. Например, объединение множеств размеров `n` и `m` в среднем требует `O(n + m)` работы.

### Когда имеет смысл создавать множество

Если нужно проверить много кандидатов по одним и тем же данным, множество полезно создать **один раз до цикла**. Не нужно пересоздавать его на каждой проверке.

В примере есть `n` запрещённых кодов и `m` кандидатов:

- Поиск каждого кандидата в исходном списке в худшем случае даёт `O(n · m)`.
- Создание множества один раз занимает в среднем `O(n)`, затем `m` проверок — в среднем `O(m)`. Вместе — `O(n + m)`.

За ускорение платим дополнительной памятью: множество из `u` разных кодов требует `O(u)` памяти. Ради одной проверки преобразовывать весь список в множество не обязательно: само преобразование тоже требует времени.

In [ ]:
banned_codes = [101, 205, 309, 101]
candidates = [205, 400, 309, 500]

banned_set = set(banned_codes)  # один раз до цикла
blocked_count = 0
for candidate in candidates:
    if candidate in banned_set:
        blocked_count += 1
print(blocked_count)  # 2

<a id="summary"></a>
## 5. Шпаргалка

| Задача | Средство |
|---|---|
| Хранить уникальные значения | `set()` |
| Убрать повторы из входных данных без требования порядка | `set(data)` |
| Найти общие элементы | `a & b` для множеств |
| Проверить все требования | `required <= available` |
| Хранить соответствие «ключ → значение» | `dict` |
| Получить значение с запасным ответом | `d.get(key, default)` |
| Перебрать ключи и значения вместе | `for key, value in d.items()` |
| Увеличить счётчик | `d[key] = d.get(key, 0) + 1` |
| Получить отдельную поверхностную копию | `.copy()` |

### Проверь понимание

1. Почему `{}` нельзя использовать как пустое множество?
2. Чем `a - b` отличается от `b - a`?
3. Чем `remove()` отличается от `discard()` у множества?
4. Что проверяет `30 in {"цена": 30}`? Как проверить наличие значения `30`?
5. Что вернёт `{"запас": 0}.get("запас", 10)`?
6. Можно ли сделать список значением словаря? А ключом?
7. Почему `(1, 2)` может быть ключом, а `(1, [2])` — нет?
8. Почему поиск по ключу обычно `O(1)`, а поиск через `.values()` — `O(n)`?

### Где искать материал для ДЗ

- Задачи 1–2: раздел 1 — создание множеств и пересечение.
- Задача 3: раздел 2 — получение значения через `.get()`.
- Задача 4: раздел 2 — перебор через `.items()` и создание нового словаря.
- Задача 5: раздел 2 — увеличение счётчика.

[Открыть домашнее задание](homework.ipynb).

### Справочные источники

Для уточнения правил: [множества](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset), [словари](https://docs.python.org/3/library/stdtypes.html#mapping-types-dict), [определение хешируемости](https://docs.python.org/3/glossary.html#term-hashable). Оценки операций CPython: [Time complexity of built-in types](https://docs.python.org/3/builtins/time-complexity.html).